# Notes on the NeRF paper

In this notebook, I summarize concepts of the paper, and explore their implementations.

In [ ]:
# Imports

# File handling
import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Json handling
import json


# Image loading
import imageio.v2 as imageio
# Plotting
import matplotlib.pyplot as plt
# Pytorch
import torch
import torch.nn as nn

In [ ]:
import sys

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))

#print(sys.executable)

In [ ]:
# Constants

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# data paths

DATA_ROOT = "./data/"

TRAINING_DATASET_DIR = DATA_ROOT + "train/"
TRAINING_JSON = "transforms_train.json"

VALIDATION_DATASET_DIR = DATA_ROOT + "val/"
VALIDATION_JSON = "transforms_val.json"

TEST_DATASET_DIR = DATA_ROOT + "test/"
TEST_JSON = "transforms_test.json"


# bounds
NEAR = 2.0
FAR = 6.0



print("Using device:", DEVICE)
print(f"Camera Bounds - near : {NEAR}, far :{FAR}")

if os.path.exists(DATA_ROOT):
    print(f"Dataset directory exists: {DATA_ROOT}")
else:
    print(f"ERROR: Training dataset directory not found: {DATA_ROOT}")

if os.path.exists(DATA_ROOT + TRAINING_JSON):
    print(f"Training json exists: {TRAINING_JSON}")
else:
    print(f"ERROR: Training json not found: {TRAINING_JSON}")


In [ ]:
# Globals

### Training scenes center and scale , used for positions normalization mentionned in positional encoding section:
with open(os.path.join(DATA_ROOT, TRAINING_JSON), "r") as f:
        training_metadata = json.load(f)
const_frame = training_metadata["frames"][0]
cam_pos = torch.tensor(const_frame["transform_matrix"], dtype=torch.float32)
cam_pos = cam_pos[:3,3]
scene_center = cam_pos.mean(0)
scene_scale = torch.norm(cam_pos - scene_center, dim=-1).max()
###

# Global index values placeholders for GT display and inference rendering

global_validation_index = 18
global_test_index = 54



# Using randomized indices:

# Validation
with open(os.path.join(DATA_ROOT, VALIDATION_JSON), "r") as f:
        val_metadata = json.load(f)

val_index = torch.randint(
0, len(val_metadata["frames"]), (1,)).item()  # Picking 1 random frame index for visual validation
globals()["global_validation_index"] = val_index

# Test
with open(os.path.join(DATA_ROOT, TEST_JSON), "r") as f:
        test_metadata = json.load(f)

t_index = torch.randint(
0, len(test_metadata["frames"]), (1,)).item()  # Picking 1 random frame index for visual validation
globals()["global_test_index"] = t_index

print("global validation index : ", global_validation_index)
print("global test index :", global_test_index)

In [ ]:
# Other value potentially shared by the 3 version (minimal, intermediary,complete = No PE + No HVS, No HVS, PE+HVS):

p_dimension = 3 # Based on version with no positional encoding dimension of positions (x)
d_dimension = 3 # Based on version with no positional encoding dimension of directions (d)
n_iterations = 2100
batch_size = 1024
chunk_size = 1024
n_coarse = 32
n_fine= 64

# Values of L for positional encoding: 
L_x = 10 # for positions
L_d = 4 # for directions


## Using MLP in the scene representation :

Static scene represented as a continuous 5D vector-valued function.

- Inputs : 2D direction $\mathbf{d} = (\theta,\phi)$ at each point $\mathbf{x} = (x,y,z)$ :  single coordinate $(x,y,z,\theta,\phi)$
- Outputs :  volume density $\sigma$ and view-dependent RGB color $\mathbf{c} = (r,g,b)$ :  $(r,g,b,\sigma)$
- That is $$F_\Theta : (\mathbf{x},\mathbf{d}) \rightarrow (\mathbf{c},\sigma)$$

## Understanding mathematical formulations: 

### <u>Rendering color with classical volume rendering :</u> 

Expected color of camera ray $ r(t) = \mathbf{o} + t\mathbf{d}$  , $\mathbf{o}$ the origin, $\mathbf{d}$ the direction of the ray.  
With near bound $t_n$ and far bound $t_f$ , with volume density $\sigma(\mathbf{x})$:
$$
C(\mathbf{r}) = \int_{t_n}^{t_f} T(t)\,\sigma(\mathbf{c}(\mathbf{r}(t)), \mathbf{d})\, dt
$$

Where : 
$$
T(t) = exp( - \int_{t_n}^t \sigma(\mathbf{r}(s))ds)
$$

Is the accumulated transmittance along the ray, from $t_n$, to $t$.

##### Looking up "deterministic quadrature" and why "it limits the MLP":

- Deterministic quadrature :  Numerical integration computed at fixed, evenly spaced points.
- From paper : *"The MLP would only be queried at a fixed discrete set of locations"*


Remedy to that by using : 

#### **Stratified sampling approach** : 

- Partionning $[t_n,t_f]$ into *$N$* evenly-spaced bins 
- Draw one sample **uniformly** at random from within each bin: $$t_i\sim\mathcal{U}\left[t_n+\frac{i-1}{N}(t_f-t_n),t_n+\frac{i}{N}(t_f-t_n)\right]$$
- Stratified sampling enables to represent continuous scene because of evalutation at *more* continuous positions over the course of optimization.
- Adds "randomness" to the approach based on Max quadratures.




In [ ]:

def stratify_sampling_vectorized(
    t_near, t_far, N, num_rays, randomized=True, device=DEVICE
):
    """
    Stratified sampling for multiple rays at once
    """
    bins = torch.linspace(t_near, t_far, N + 1, device=device)

    bin_widths = bins[1:] - bins[:-1]

    
    u = torch.rand(num_rays, N, device=device)
    t_samples = bins[:-1] + u * bin_widths

    if not randomized: # Deterministic sampling for inference, found while browsing in original codebase
        t_samples = bins[:-1] + 0.5 * bin_widths
        t_samples = t_samples.expand(num_rays, N)
    
    return t_samples

### <u>Estimating the continuous scene represention using **discrete computations**</u> :

Max (1995) Quadrature rule cited and reused :

- Enables description of emission and absorption of light along a ray accurately with a **finite number of samples**
- Gives us an estimated color along the ray: $$ \hat{C}(\mathbf{r}) = \sum_{i=1}^{N} T_i(1-exp(-\sigma_i\delta_i))\mathbf{c}_i$$
- With the transmitance $T_i$ : $$T_i = exp\left(-\sum_{j=1}^{i-1}\sigma_j\delta_j\right)$$
- And $\delta_i = t_{i+1}-t_i$, distance between adjacent samples
- Paper also precises we can write $$\hat{C}(\mathbf{r}) = \sum_{i=1}^{N} T_i\alpha_i\mathbf{c}_i$$ with $\alpha_i = 1-exp(-\sigma_i\delta_i)$ to reduce it to traditional alpha compositing.


<!-- $$
T(t) = exp( - \int_{t_n}^t \sigma(\mathbf{r}(s))ds)
$$ -->

In [ ]:
def volume_rendering(rays_o, rays_d, model, N, t_vals, encoding_fn=None):# stratified sampling now out of the function as it is now also used for fine sampling as well
    
    # get ray count:
    num_rays = rays_o.shape[0]


    # model inputs

    # normalizing rays_d
    rays_d = rays_d / torch.norm(rays_d, dim=-1, keepdim=True)

    # points along the ray r(t) = o + t*d (x in the article)
    preencoded_points = rays_o[:, None, :] + t_vals[..., None] * rays_d[:, None, :]

    preencoded_directions = rays_d[:, None, :].expand_as(preencoded_points)

    points = preencoded_points
    directions = preencoded_directions
    
    # if positional encoding function passed : encode points and directions before passing to model
    # Retroactively added there through function passed as argument, positional encoding is treated much later in the notebook
    #MODIF(1) 
    if callable(encoding_fn):

        preencoded_points = (preencoded_points-scene_center) / scene_scale
        points = encoding_fn(preencoded_points,L_x)
        directions = encoding_fn(preencoded_directions,L_d)


    B,N,Cpos = points.shape
    _,_,Cdir = directions.shape

    points = points.reshape(B*N,Cpos)
    directions = directions.reshape(B*N,Cdir)

    rgb, sigmas = model(
        points,directions
    )  # As a reminder, the model is supposed to return emitted colors, and volume densities

    rgb = rgb.reshape(num_rays, N, 3)
    sigmas = sigmas.reshape(num_rays, N)


    deltas = t_vals[:, 1:] - t_vals[:, :-1]
    # give same length to deltas than sigmas
    deltas = torch.cat(
        [deltas, deltas[:, -1:]],
        dim=1,
    )


    torch.clamp(sigmas, max=50.0)

    T = torch.exp(
        -torch.cat(
            [
                torch.zeros((num_rays, 1), device=DEVICE, dtype=sigmas.dtype),
                torch.cumsum(sigmas * deltas, dim=1)[:, :-1],
            ],
            dim=1,
        )
    )
    alphas = 1 - torch.exp(-sigmas * deltas)

    weights = T*alphas # weights needed for the fine network

    C = torch.sum((T * alphas)[..., None] * rgb, dim=1)



    return C, weights

## Sampling rays from the dataset


#### Getting the rays from sets of images and camera intrinsics and extrinsics:

Again : ray $ r(t) = \mathbf{o} + t\mathbf{d}$  , $\mathbf{o}$ the origin, $\mathbf{d}$ the direction of the ray.

I will only focus on working with the **synthetic sets of pictures**, rendered from **Blender** scenes. 

**-->** Had to look-up the original project as I didn't recognize what was given in the json

In [ ]:
# Get_rays function cattered around the blender dataset


def get_rays(height, width, focal, camera_pose):
    """
    Get rays origin and direction from a single image

    :param height,width: dimensions of the image in pixels
    :param focal: should be 0.5 * width/tan(0.5*camera_angle_x),  with camera_angle_x, the horizontal fov from json files of the blender dataset
    :param camera_pose: camera_to_world transform_matrix from json files of the blender dataset
    """

    # pixel grid to work with i for width, j for height
    i, j = torch.meshgrid(
        torch.arange(width, device=DEVICE, dtype=torch.float32),
        torch.arange(height, device=DEVICE, dtype=torch.float32),
        indexing="xy",
    )

    # directions of rays , Blender camera looks along -z
    directions = torch.stack(
        [(i - width * 0.5) / focal, -(j - height * 0.5) / focal, -torch.ones_like(i)],
        dim=-1,
    ) 

    camera_origin = camera_pose[:3, 3]  # Camera translation part of transform matrix
    camera_rotation = camera_pose[:3, :3]
    camera_rotation = camera_rotation.to(DEVICE)

    rays_d = torch.sum(directions[..., None, :] * camera_rotation, dim=-1)
    # direction normalization
    rays_d = rays_d / torch.norm(rays_d, dim=-1, keepdim=True)

    # repeating view of rays origin for batching purposes
    rays_o = camera_origin.expand_as(rays_d)

    return rays_o, rays_d

#### Let's now work around loading metadata and the actual images from the blender dataset

In [ ]:


def get_blender_metadata(json_file):
    """
    Loads metadata for the Blender synthetic dataset

    :param json_file (str): name of the json file associated to the images folder

    :return { frames, height, width, focal, n_frames}: directory path, frames (pictures), resolution width and heights of pictures, focal length, number of frames



    """
    with open(os.path.join(DATA_ROOT, json_file), "r") as f:
        metadata = json.load(f)

    # retrieving metadata
    frames = metadata["frames"]
    camera_angle_x = metadata["camera_angle_x"]

    # retrieving resolution from one frame
    first_frame_path = os.path.join(
        DATA_ROOT, (frames[0]["file_path"]).lstrip("./") + ".png"
    )
    first_frame = imageio.imread(first_frame_path, pilmode="RGB")
    height, width = first_frame.shape[:2]

    # compute focal length from camera_angle_x:
    camera_angle_x_tensor = torch.tensor(camera_angle_x, dtype=torch.float32)
    focal = 0.5 * width / torch.tan(0.5 * camera_angle_x_tensor)

    return {
        "frames": frames,
        "height": height,
        "width": width,
        "focal": focal,
        "n_frames": len(frames), 
    }


def load_blender_training_frame(metadata, index):
    """
    Loads a single image (frame) from the Blender synthetic dataset and prepares for rays and rgb retrieval

    :param json_file (str): json metadata file associated to frame with camera characteristics
    :param index (int): index of image to load
    """
    frame = metadata["frames"][index]
    height = metadata["height"]
    width = metadata["width"]
    focal = metadata["focal"]

    image_path = os.path.join(DATA_ROOT, frame["file_path"] + ".png")
    image = imageio.imread(image_path, pilmode="RGB")
    image = torch.tensor(image, dtype=torch.float32) / 255.0  # Normalizing the image

    camera_pose = torch.tensor(frame["transform_matrix"], dtype=torch.float32)

    rays_o, rays_d = get_rays(height, width, focal, camera_pose)

    rays_o = rays_o.to(DEVICE)
    rays_d = rays_d.to(DEVICE)

    # Flatten tensors for per-pixel correspondence with target RGB colors:
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    target_rgb = image.reshape(-1, 3)
    target_rgb = target_rgb.to(DEVICE)

    return rays_o, rays_d, target_rgb


def load_blender_rendering_rays(metadata, index):

    frame = metadata["frames"][index]
    height = metadata["height"]
    width = metadata["width"]
    focal = metadata["focal"]

    camera_pose = torch.tensor(frame["transform_matrix"], dtype=torch.float32)

    rays_o, rays_d = get_rays(height, width, focal, camera_pose)

    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)

    return rays_o, rays_d

#### Providing the network architecture :

The model architecture, as it is described in the article.

<!-- add a figure there corresponding to the current architecture -->

In [ ]:
class NeRF(nn.Module):
    def __init__(
        self, p_dimension, d_dimension, hidden_dimension=256, num_layers=8, output_dimension=3
    ):
        super().__init__()


        self.layers = nn.ModuleList()

        for i in range(num_layers):
            in_dim = p_dimension if i == 0 else hidden_dimension + p_dimension if i==4 else hidden_dimension # skip at fifth layer
            self.layers.append(nn.Linear(in_dim, hidden_dimension))
        
        self.head = nn.Linear(hidden_dimension, hidden_dimension +1 )
        
        self.rgb_head = nn.Sequential(
            nn.Linear(hidden_dimension+d_dimension, hidden_dimension // 2),
            nn.ReLU(True),
            nn.Linear(hidden_dimension // 2, 3), # ReLU layer with 128 channels as mentionned in paper (256/2)
        )
        

        nn.init.constant_(self.head.bias, 0.1)  # non-zero bias for scene initialization 

        self.output_activation = nn.Sigmoid()

    def forward(self, positions, directions):
        """
        Input:
            positions: [..., p_dimension]
            directions: [..., d_dimension]
        Output:
            rgb : [...,3]
            sigma : [...]
        """
        h = positions

        for i, layer in enumerate(self.layers):
            if i == 4:
                h = torch.cat([h, positions],dim=-1)
            h = torch.relu(layer(h))
        
        out = self.head(h)
        sigma = torch.relu(out[...,0])
        features = out[...,1:]
        rgb_input = torch.cat([features,directions],dim = -1)
        rgb = self.rgb_head(rgb_input)
        rgb = self.output_activation(rgb)
        return rgb, sigma.squeeze(-1)  # sigma : [...]

#### Then we need to compute the **loss** :

The paper mentions a SSE (sum of squared errors) between estimated (rendered) and ground truth pixel colors. 

When hierarchical sampling will be added, the fine part should be included, as of now, we only have the coarse one
$$\mathcal{L} = \sum_{r \in \mathcal{R}}\left\|\hat{C}(r) - C(r)\right\|_2^2$$

In [ ]:
# SSE error from MSE with reduction = sum :
# from pytorch documentation : "The division by NN can be avoided if one sets reduction = 'sum'."

# criterion = torch.nn.MSELoss(reduction='sum')
# sse = criterion(estimated_colors, targeted_colors)

#### Then we can define a training loop:

Training is done on a random sample of rays : *" At each optimization iteration, we randomly sample
a batch of camera rays from the set of all pixels in the dataset,"*

In [ ]:
# training is done only on random sample of rays


def sample_random_rays(rays_o, rays_d, target_rgb, batch_size):
    """
    returns a random batch of rays
    """
    num_rays = rays_o.shape[0]
    indices = torch.randint(0, num_rays, (batch_size,))

    return rays_o[indices], rays_d[indices], target_rgb[indices]

#### A few other helper functions for the training loop :

___
___

In [ ]:
# Helper functions for ground truth image retrieval and predicted image generation : 

def get_GT_and_metadata(dataset_json, index):
    """
    Get a ground truth image and the metadata necessary to generate an image from the camera poses, corresponding to the ground truth.
    
    Used for the validation and test datasets
    
    :param dataset_dir (str): path to the dataset directory containing the images
    :param dataset_json (str): path to the json containing metadata on related pictures

    :return {gt_img, metadata, index} : The loaded ground truth image, metadata of the folder and index corresponding to the ground truth image
    """
    metadata = get_blender_metadata(dataset_json)


    frame_path = metadata["frames"][index]

    img_path = os.path.join(DATA_ROOT, frame_path["file_path"].lstrip("./") + ".png")

    gt_img = imageio.imread(img_path, pilmode="RGB")
    gt_img = torch.tensor(gt_img, dtype=torch.float32) / 255.0
    gt_img = gt_img.cpu().numpy()


    return gt_img, metadata



def generate_predicted(metadata, index, coarse_model, n_coarse=64, near=NEAR, far=FAR,chunk_size=1024,encoding_fn=None, hierarchical_sampling_fn=None,fine_model=None,n_fine=128):
    """
    Renders a full image for visualisation 
    """

    with torch.no_grad():

        height, width = metadata["height"], metadata["width"]

        rays_o, rays_d = load_blender_rendering_rays(
            metadata, index
        )
        rays_o = rays_o.to(DEVICE)
        rays_d = rays_d.to(DEVICE)


        # Generate in chunks so that there is no spike of VRAM use:

        num_rays = rays_o.shape[0]


        all_predicted_colors = []

        for i in range(0,num_rays,chunk_size):

            rays_o_chunk = rays_o[i:i+chunk_size]
            rays_d_chunk = rays_d[i:i+chunk_size]

            # coarse sampling:
            t_coarse = stratify_sampling_vectorized(near,far,n_coarse,rays_o_chunk.shape[0],randomized=False,device=DEVICE)

            chunk_predicted_coarse_colors,chunk_coarse_weights = volume_rendering(
                rays_o=rays_o_chunk,
                rays_d=rays_d_chunk,
                model=coarse_model,
                N=n_coarse,
                t_vals=t_coarse,
                encoding_fn= encoding_fn,
            )
            
            if callable(hierarchical_sampling_fn): #MODIF(2)
                t_fine = hierarchical_sampling_fn(chunk_coarse_weights,t_coarse,n_fine, randomized=False)

                t_all = torch.cat([t_coarse, t_fine], dim=-1)
                t_all, _ = torch.sort(t_all, dim=-1)

                

                chunk_predicted_fine_colors, _ = volume_rendering(
                    rays_o_chunk,
                    rays_d_chunk,
                    fine_model,
                    (n_coarse+n_fine),
                    t_all,
                    encoding_fn= encoding_fn,
                )
                all_predicted_colors.append(chunk_predicted_fine_colors)
            else: # If no hierarchical sampling :  
                all_predicted_colors.append(chunk_predicted_coarse_colors)

        predicted_colors = torch.cat(all_predicted_colors, dim=0)

        predicted_img = (
            predicted_colors.reshape(height, width, 3)
            .cpu()
            .clamp(0, 1)
            .numpy()
        )
    return predicted_img

In [ ]:
# Show comparison of ground truth and rendered images for visual validation, and test dataset
def display_comparison(gt_img, rendered_img, index, iteration = None ,category="Validation",psnr=None,version_description=""):
    fig = plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(gt_img)
    plt.title(f"{category} Ground Truth - Frame {index}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(rendered_img)
    additional = ""
    if iteration : additional = f"- Iteration {iteration}"
    plt.title(f"{category} Rendered {additional}")
    if psnr : fig.text(0.5,0.01,f"PSNR = {psnr:.2f}\n {version_description}",ha="center",fontsize=12)
    plt.axis("off")

    plt.show()

In [ ]:
#https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio
def get_PSNR(gt_img, rendered_img, epsilon = 1e-10): # really small epsilon to avoid dividing by 0
    # from np imgs to representation as tensors:
    gt_img = torch.as_tensor(gt_img,dtype=torch.float32)
    rendered_img = torch.as_tensor(rendered_img,dtype=torch.float32)

    mse = torch.mean((rendered_img-gt_img)**2)
    psnr = 10.0*torch.log10(1.0/(mse+epsilon))
    return psnr

___
___
Back to following the paper:

#### The actual training loop :

In [ ]:
def train_NeRF(
    coarse_model,
    n_iterations=3000,
    batch_size=1024,
    learning_rate=5e-4,
    N_coarse=64,
    near=NEAR,
    far=FAR,
    device=DEVICE,
    chunk_size = 1024,
    hierarchical_sampling_fn=None,
    fine_model=None, # Keep None to disable hierarchical sampling , again, retroactively added for something treated much later
    N_fine=128,
    encoding_fn=None, # keep None to disable positional encoding
    version_description=""
        
):
    coarse_model=coarse_model.to(device)
    # paper mentions using Adam optimizer :
    optimizer = torch.optim.Adam(coarse_model.parameters(), lr=learning_rate)

    if callable(hierarchical_sampling_fn):
        fine_model=fine_model.to(device)
        # hierarchical sampling, replace optimizer with full version :
        optimizer = torch.optim.Adam(list(coarse_model.parameters()) + list(fine_model.parameters()), lr=learning_rate)


    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer,
        gamma=0.999
    )

    loss_function = torch.nn.MSELoss(reduction="sum")  # SSE

    train_losses = []
    validation_losses = []
    PSNR_values = []

    chunk_size = chunk_size

    training_metadata = get_blender_metadata(TRAINING_JSON)
    
    validation_index = globals()["global_validation_index"]
    validation_gt_img, validation_metadata  = get_GT_and_metadata( VALIDATION_JSON,validation_index)

    height, width = validation_metadata["height"], validation_metadata["width"]

    for iteration in range(n_iterations):

        i = torch.randint(0, training_metadata["n_frames"], (1,)).item()
        rays_o, rays_d, target_rgb = load_blender_training_frame(training_metadata, i)

        # Random batch of rays from the selected frame
        rays_o_batch, rays_d_batch, targeted_rgb_batch = sample_random_rays(
            rays_o, rays_d, target_rgb, batch_size
        )

        # Stratified sampling for t_coarse:
        t_coarse = stratify_sampling_vectorized(
            near, far, N_coarse, rays_o_batch.shape[0], randomized=True, device=DEVICE
        )  # shape : [num_rays,N]
        
        coarse_colors, coarse_weights = volume_rendering(
            rays_o_batch, rays_d_batch, coarse_model, N_coarse, t_vals=t_coarse,encoding_fn=encoding_fn)
        
        # Loss computation 1/2
        loss = loss_function(coarse_colors, targeted_rgb_batch)

        # Again, hierarchical sampling retroactively added :

        if callable(hierarchical_sampling_fn): #MODIF(2)
            # Fine sampling 
            t_fine = hierarchical_sampling_fn(coarse_weights=coarse_weights,t_coarse=t_coarse,N_fine=N_fine, randomized=True)
            
            t_all = torch.cat([t_coarse,t_fine],dim=-1)
            #torch.cuda.synchronize() # for debugging purposes, slows down otherwise
            t_all, _ = torch.sort(t_all, dim=-1)

            fine_colors, fine_weights = volume_rendering(
                rays_o_batch, rays_d_batch, fine_model, N=(N_coarse+N_fine), t_vals=t_all,encoding_fn=encoding_fn)
            # Loss computing 2/2 complete coarse loss with fine loss part
            loss = loss + loss_function(fine_colors, targeted_rgb_batch)


        # Backpropagation
        optimizer.zero_grad(set_to_none=True) # sets grads to None instead of 0  --> https://docs.pytorch.org/docs/2.11/generated/torch.optim.Optimizer.zero_grad.html
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

        if iteration % 50 == 0:
            # Random ray batch validation
            coarse_model.eval()
            if callable(hierarchical_sampling_fn): fine_model.eval()
            with torch.no_grad():
                sse_val_index = torch.randint(
                    0, validation_metadata["n_frames"], (1,)
                ).item()
                sse_val_rays_o, sse_val_rays_d, sse_val_rays_rgb = (
                    load_blender_training_frame(validation_metadata, sse_val_index)
                )
                sse_val_rays_o_batch, sse_val_rays_d_batch, sse_val_rays_rgb_batch = (
                    sample_random_rays(
                        sse_val_rays_o, sse_val_rays_d, sse_val_rays_rgb, batch_size
                    )
                )

                # coarse
                t_coarse = stratify_sampling_vectorized(near, far, N_coarse, sse_val_rays_o_batch.shape[0], randomized=False, device=DEVICE)

                sse_val_coarse_predicted, sse_val_coarse_weights = volume_rendering(
                    rays_o=sse_val_rays_o_batch,
                    rays_d=sse_val_rays_d_batch,
                    model=coarse_model,
                    N=N_coarse,
                    t_vals=t_coarse,
                    encoding_fn=encoding_fn
                )

                sse_val_loss = loss_function(sse_val_coarse_predicted, sse_val_rays_rgb_batch)

                if callable(hierarchical_sampling_fn): #MODIF(2)
                    t_fine = hierarchical_sampling_fn(coarse_weights=sse_val_coarse_weights,t_coarse=t_coarse,N_fine=N_fine, randomized = False)

                    t_all = torch.cat([t_coarse,t_fine],dim=-1)
                    t_all, _ = torch.sort(t_all, dim=-1)

                    sse_val_fine_predicted, _ = volume_rendering(
                        rays_o=sse_val_rays_o_batch,
                        rays_d=sse_val_rays_d_batch,
                        model=fine_model,
                        N=(N_coarse+N_fine),
                        t_vals=t_all,
                        encoding_fn=encoding_fn
                    )
                    sse_val_loss = loss_function(sse_val_fine_predicted, sse_val_rays_rgb_batch)

                validation_losses.append(sse_val_loss.item())
            coarse_model.train()
            if callable(hierarchical_sampling_fn): fine_model.train()

        # Visual validation
        if iteration % 100 == 0:
            if callable(hierarchical_sampling_fn): fine_model.eval()
            coarse_model.eval()

            validation_predicted_img = generate_predicted(metadata = validation_metadata, 
                                                          index = validation_index,
                                                          coarse_model = coarse_model,
                                                          fine_model = fine_model,
                                                          n_coarse = N_coarse,
                                                          n_fine = N_fine,
                                                          near = NEAR, 
                                                          far = FAR,
                                                          encoding_fn=encoding_fn,
                                                          hierarchical_sampling_fn=hierarchical_sampling_fn)
            
            with torch.no_grad():
                psnr = get_PSNR(gt_img=validation_gt_img ,rendered_img=validation_predicted_img,)
                PSNR_values.append(psnr)
                display_comparison(validation_gt_img, validation_predicted_img, validation_index, iteration = iteration ,category="Validation", psnr=psnr,version_description=version_description)
                
            coarse_model.train()
            if callable(hierarchical_sampling_fn): fine_model.train()
    print("Training over")

    return train_losses, validation_losses, PSNR_values
    
    

#### We can now already try it for a baseline version of NeRF

(Please note that by "baseline", we are not doing full ablation like in the paper, as view dependence is always present)

In [ ]:
def plot_values_evolution(n_iterations,first_value,first_value_label="",second_value=None,second_value_label="", value_type="SSE Loss"):
    plt.figure(figsize=(8, 5))
    x1= [i * n_iterations / (len(first_value)-1) for i in range(len(first_value))]
    plt.plot(x1,first_value, label=f"{first_value_label} {value_type}") #label="Training SSE Loss"
    title_second_part=""
    if second_value is not None:
        x2= [i * n_iterations / (len(second_value)-1) for i in range(len(second_value))]
        plt.plot(x2, second_value, label=f"{second_value_label} {value_type}") #label="Validation SSE Loss"
        title_second_part = f"and {second_value_label}"
    plt.xlabel("Iteration")
    plt.ylabel(f"{value_type}")
    plt.title(f"{first_value_label} {title_second_part} Evolution")
    plt.legend()
    plt.show()

In [ ]:
def display_test_render_comparison(coarse_model,
                                   n_coarse=32,
                                   test_index=global_test_index,
                                   near= NEAR,
                                   far= FAR,
                                   encoding_fn=None,
                                   hierarchical_sampling_fn=None,
                                   fine_model=None,
                                   n_fine=64,
                                   version_description=""
                                   ):

    test_gt_img, test_metadata = get_GT_and_metadata(TEST_JSON, test_index)
    test_predicted_img = generate_predicted(
                            metadata = test_metadata, 
                            index = test_index,
                            coarse_model = coarse_model,
                            fine_model = fine_model,
                            n_coarse = n_coarse,
                            n_fine = n_fine,
                            near = NEAR, 
                            far = FAR,
                            encoding_fn=encoding_fn,
                            hierarchical_sampling_fn=hierarchical_sampling_fn
                            )
    psnr=get_PSNR(gt_img=test_gt_img,rendered_img=test_predicted_img)
    display_comparison(gt_img=test_gt_img,
                       rendered_img=test_predicted_img,
                       index=test_index,
                       iteration=None,
                       category="TEST",
                       psnr=psnr,
                       version_description=version_description)

In [ ]:
p_dimension_min = p_dimension
d_dimension_min = d_dimension
n_iterations_min= n_iterations
batch_size_min = batch_size
n_coarse_min= n_coarse
chunk_size_min= chunk_size

version_description_min = "no positional encoding, no hierarchical volume sampling"
minimal_model = NeRF(p_dimension=p_dimension_min,
               d_dimension=d_dimension_min,
               ).to(DEVICE)


training_losses_min, validation_losses_min, psnr_values_min= train_NeRF(
    coarse_model=minimal_model,
    n_iterations=n_iterations_min,
    batch_size=batch_size_min,
    learning_rate=5e-4,
    N_coarse=n_coarse_min,
    near=NEAR,
    far=FAR,
    device=DEVICE,
    chunk_size = chunk_size_min,
    hierarchical_sampling_fn=None, 
    fine_model=None, 
    N_fine=n_fine,
    encoding_fn=None,
    version_description=version_description_min
    
)

plot_values_evolution(
    n_iterations=n_iterations_min,
    first_value=training_losses_min,
    first_value_label="Training",
    second_value=validation_losses_min,
    second_value_label="Validation",
    value_type = "SSE Loss",
    )

plot_values_evolution( # Not a loss, but plotting of PSNR
    n_iterations=n_iterations_min,
    first_value=psnr_values_min,
    first_value_label="Validation",
    value_type = "PSNR",
)

display_test_render_comparison(coarse_model=minimal_model,
                                   n_coarse=n_coarse_min,
                                   test_index=global_test_index,
                                   near= NEAR,
                                   far= FAR,
                                   encoding_fn=None,
                                   hierarchical_sampling_fn=None,
                                   fine_model=None,
                                   n_fine=n_fine,
                                   version_description=version_description_min
                                   )


#### Adding positional encoding

In the paper, the encoding function is defined as the following mapping : 

$$
\gamma(p) = (sin(2^0\pi p),cos(2^0\pi p), \dots , sin(2^{L-1}\pi p),cos(2^{L-1}\pi p))
$$

It is applied to each of the three coordinates of $\mathbf{x}$ and each of the three component of $\mathbf{d}$

With $L = 10$ for $\gamma(\mathbf{x})$ and $L = 4$ for $\gamma(\mathbf{d})$

In [ ]:
def positional_encoding(x,L):
    freq = 2**torch.arange(L, device=x.device )*torch.pi
    x_encoded = x.unsqueeze(-1) * freq
    x_sin_part = torch.sin(x_encoded)
    x_cos_part = torch.cos(x_encoded)
    # interleaving sin and cos (only aesthetic)
    x_encoded = torch.stack([x_sin_part,x_cos_part],dim=-1)
    x_encoded = x_encoded.flatten(-2).flatten(-2)
    return x_encoded

As the positional encoding affects $\mathbf{x}$ and $\mathbf{d}$, we are going to adapt the volume rendering function to apply this new mapping, search for comment #MODIF(1) 

Let's check it:


In [ ]:
L_x_intrmd= L_x
L_d_intrmd= L_d

p_dimension_intrmd = p_dimension * L_x_intrmd*2 #(time 2 because of L_x*sine terms + L_x*cosine terms) # should be 3*10*2=60
d_dimension_intrmd = d_dimension * L_d_intrmd*2 # should be 3*4*2=24
n_iterations_intrmd= n_iterations
batch_size_intrmd = batch_size
n_coarse_intrmd= n_coarse
chunk_size_intrmd= chunk_size
encoding_fn_intrmd=positional_encoding # We are actually using the positional encoding function here

version_description_intrmd ="positional encoding, no hierarchical volume sampling"

intrmd_model = NeRF(p_dimension=p_dimension_intrmd,
               d_dimension=d_dimension_intrmd,
               ).to(DEVICE)


training_losses_intrmd, validation_losses_intrmd,psnr_values_intrmd= train_NeRF(
    coarse_model=intrmd_model,
    n_iterations=n_iterations_intrmd,
    batch_size=batch_size_intrmd,
    learning_rate=5e-4,
    N_coarse=n_coarse_intrmd,
    near=NEAR,
    far=FAR,
    device=DEVICE,
    chunk_size = chunk_size_intrmd,
    hierarchical_sampling_fn=None, 
    fine_model=None, 
    N_fine=n_fine,
    encoding_fn=encoding_fn_intrmd,
    version_description=version_description_intrmd
    
)

plot_values_evolution(
    n_iterations=n_iterations_intrmd,
    first_value=training_losses_intrmd,
    first_value_label="Training",
    second_value=validation_losses_intrmd,
    second_value_label="Validation",
    value_type = "SSE Loss",
    )

plot_values_evolution( # Not a loss, but plotting of PSNR
    n_iterations=n_iterations_intrmd,
    first_value=psnr_values_intrmd,
    first_value_label="Validation",
    value_type = "PSNR",
)

display_test_render_comparison(coarse_model=intrmd_model,
                                   n_coarse=n_coarse_intrmd,
                                   test_index=global_test_index,
                                   near= NEAR,
                                   far= FAR,
                                   encoding_fn=encoding_fn_intrmd,
                                   hierarchical_sampling_fn=None,
                                   fine_model=None,
                                   n_fine=n_fine,
                                   version_description=version_description_intrmd
                                   )







#### Adding hierarchical volume sampling

The final improvement in the article is the hierarchical sampling, which allows for further sampling in areas of the scene where important content is expected. By "expected", what I mean is that these areas are found using **inverse transform sampling** on the weights issued from a first evaluation on a first *coarse* sampling, then new samples *(fine sampling)* are added around these points of interests to constitute, together with the base sampling, a set of points to evaluate a **fine network** as opposed to the one we have been working with up until now.

At this point in the paper, a reminder on (and rewrite of) the computation of the color is given for the coarse network as : $$\hat{C_c}(\mathbf{r}) = \sum_{i=1}^{N_c} w_i c_i$$ 


With $$ w_i = T_i\alpha_i$$ $$ w_i = T_i(1-exp(-\sigma_i\delta_i))$$ the weights used later for the inverse transform sampling


This time, the loss is identical to the one mentionned in the paper: 
$$
\mathcal{L} =\sum_{r \in \mathcal{R}}\left[\left\|\hat{C}_c(r) - C(r)\right\|_2^2+\left\|\hat{C}_f(r) - C(r)\right\|_2^2\right]$$


##### Inverse transform sampling helper function

In [ ]:

def hierarchical_sampling(coarse_weights, t_coarse, N_fine, randomized= True):


    weights = coarse_weights[...,1:-1]
    bins = 0.5 * (t_coarse[...,:-1]+t_coarse[...,1:]) # t_coarse midpoints delimit bins
    epsilon = 1e-5 # use of epsilon to avoid NaNs

    pdf = weights + epsilon # preventing NaNs
    pdf = pdf/ torch.sum(pdf, dim=-1, keepdim=True) #piecewise-constant PDF along the ray
    
    # Inverse transform sampling steps
    # https://en.wikipedia.org/wiki/Inverse_transform_sampling
    # https://pbr-book.org/4ed/Monte_Carlo_Integration/Sampling_Using_the_Inversion_Method

    # computing the quantile function of the pdf or cdf (cumulative distribution function)
    cdf = torch.cumsum(pdf,dim=-1) # cumulative distribution function
    cdf = torch.cat([torch.zeros_like(cdf[...,:1]),cdf], dim=-1)

    u = torch.rand(weights.shape[:-1]+ (N_fine,),device=weights.device)
    if not randomized: # replace u with deterministic version for inference
        u = (torch.arange(N_fine,device=weights.device)+0.5)/N_fine
        u = u.expand(weights.shape[:-1]+(N_fine,))
        u = u.contiguous()

    # Inversion of CDF by search + interpolation :
    indices = torch.searchsorted(cdf, u, right=True) #


    # Because the CDF is issued from piecewise-constant PDF, we approximate linearity locally and use the linear interpolation formula (x-x0)/(x1-x0)=(y-y0)/(y1 -y0)
    # to sample "continuous" positions 

    below = torch.clamp(indices -1, 0, cdf.shape[-1] - 1)
    above = torch.clamp(indices, 0, cdf.shape[-1] - 1)

    cdf_below =  torch.gather(cdf, -1, below)
    cdf_above =  torch.gather(cdf, -1, above)



    t_below =  torch.gather(bins, -1, below)
    t_above =  torch.gather(bins, -1, above)


    denominator = cdf_above-cdf_below
    denominator =  torch.clamp(denominator,min=epsilon) # prevents division by 0 and NaNs
    
    t_fine = t_below + (u - cdf_below)/denominator*(t_above-t_below)
    t_fine = torch.where(denominator > epsilon, t_fine, t_below)


    return t_fine


Again, a few functions above were modified to take hierarchical volume sampliing into account. These parts can be found at #MODIF(2)


Let's check it:

In [ ]:
L_x_complt= L_x
L_d_complt= L_d

p_dimension_complt = p_dimension * L_x_complt*2 #(time 2 because of L_x*sine terms + L_x*cosine terms) # should be 3*10*2=60
d_dimension_complt = d_dimension * L_d_complt*2 # should be 3*4*2=24
n_iterations_complt= n_iterations
batch_size_complt = batch_size
n_coarse_complt= n_coarse
n_fine_complt= n_fine
chunk_size_complt= chunk_size
encoding_fn_complt = positional_encoding # We are actually using the positional encoding function here
hierarchical_sampling_fn_complt = hierarchical_sampling

version_description_complt = "positional encoding + hierarchical volume sampling"


complt_coarse_model = NeRF(p_dimension=p_dimension_complt,
               d_dimension=d_dimension_complt,
               ).to(DEVICE)

complt_fine_model = NeRF(p_dimension=p_dimension_complt,
               d_dimension=d_dimension_complt,
               ).to(DEVICE)


training_losses_complt, validation_losses_complt, psnr_values_complt= train_NeRF(
    coarse_model=complt_coarse_model,
    n_iterations=n_iterations_complt,
    batch_size=batch_size_complt,
    learning_rate=5e-4,
    N_coarse=n_coarse_complt,
    near=NEAR,
    far=FAR,
    device=DEVICE,
    chunk_size = chunk_size_complt,
    hierarchical_sampling_fn=hierarchical_sampling_fn_complt, 
    fine_model=complt_fine_model, 
    N_fine=n_fine_complt,
    encoding_fn=encoding_fn_complt,
    version_description=version_description_complt
    
)

plot_values_evolution(
    n_iterations=n_iterations_complt,
    first_value=training_losses_complt,
    first_value_label="Training",
    second_value=validation_losses_complt,
    second_value_label="Validation",
    value_type = "SSE Loss",
    )

plot_values_evolution( # Not a loss, but plotting of PSNR
    n_iterations=n_iterations_complt,
    first_value=psnr_values_complt,
    first_value_label="Validation",
    value_type = "PSNR",
)

display_test_render_comparison(coarse_model=complt_coarse_model,
                                   n_coarse=n_coarse_complt,
                                   test_index=global_test_index,
                                   near= NEAR,
                                   far= FAR,
                                   encoding_fn=encoding_fn_complt,
                                   hierarchical_sampling_fn=hierarchical_sampling_fn_complt,
                                   fine_model=complt_fine_model,
                                   n_fine=n_fine_complt,
                                   version_description=version_description_complt
                                   )





